# Etiquetado de fallas y RUL

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.utils import resample

sns.set_style('whitegrid')

In [ ]:
df = pd.read_csv('../data/raw/MetroPT3.csv',
                parse_dates=['timestamp'])
df.drop(columns=['Unnamed: 0'], inplace = True)
df = df.sort_index()
df.head()

In [ ]:
failure_events = [
    ('2020-04-18 00:00', '2020-04-18 23:59'),
    ('2020-05-29 23:30', '2020-05-30 06:00'),
    ('2020-06-05 10:00', '2020-06-05 23:59'),
    ('2020-07-15 14:30', '2020-07-15 19:00')
]

df['failure'] = 0
for start, end in failure_events:
    start_ts = pd.to_datetime(start)
    end_ts = pd.to_datetime(end)

    mascara = (df['timestamp'] >= start_ts) & (df['timestamp'] <= end_ts)
    df.loc[mascara, 'failure'] = 1

df.head()

## 1. Resumen del dataset

In [ ]:
print(f"Shape: {df.shape}")
print(f"\nRango temporal: {df['timestamp'].min()} → {df['timestamp'].max()}")
print(f"\nValores nulos:\n{df.isnull().sum()}")
print(f"\nRegistros con falla: {df['failure'].sum()} ({df['failure'].mean()*100:.2f}%)")
df.describe().T.style.background_gradient(cmap='Blues')

## 2. Balance de clases

In [ ]:
counts = df['failure'].value_counts()
labels = ['Normal (0)', 'Falla (1)']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(labels, counts.values, color=['steelblue', 'tomato'], edgecolor='white')
axes[0].set_title('Conteo de registros por clase', fontweight='bold')
axes[0].set_ylabel('Registros')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=labels, autopct='%1.2f%%',
            colors=['steelblue', 'tomato'], startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Proporción de clases', fontweight='bold')

plt.suptitle('Desbalance de clases', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()

## 3. Series de tiempo con ventanas de falla

In [ ]:
sensors = ['TP2', 'TP3', 'Oil_temperature', 'Motor_current']
fig, axes = plt.subplots(len(sensors), 1, figsize=(16, 12), sharex=True)

df_plot = df.set_index('timestamp').resample('1h').mean().reset_index()

for ax, sensor in zip(axes, sensors):
    ax.plot(df_plot['timestamp'], df_plot[sensor], linewidth=0.8, color='steelblue')
    for start, end in failure_events:
        ax.axvspan(pd.to_datetime(start), pd.to_datetime(end),
                   alpha=0.3, color='tomato', label='Falla' if start == failure_events[0][0] else '')
    ax.set_ylabel(sensor, fontweight='bold')
    ax.grid(True, alpha=0.3)

axes[0].legend(loc='upper right')
axes[-1].set_xlabel('Fecha')
plt.suptitle('Sensores analógicos en el tiempo (media horaria)\nVentanas de falla en rojo',
             fontsize=13, fontweight='bold')
plt.tight_layout()

## 4. Distribución de variables analógicas: Normal vs Falla

In [ ]:
analog_cols = ['TP2', 'TP3', 'H1', 'DV_pressure', 'Reservoirs',
               'Oil_temperature', 'Motor_current', 'Caudal_impulses']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
palette = {0: 'steelblue', 1: 'tomato'}

for ax, col in zip(axes, analog_cols):
    sns.boxplot(data=df, x='failure', y=col, ax=ax,
                palette=palette, width=0.5, fliersize=1)
    ax.set_title(col, fontweight='bold')
    ax.set_xticklabels(['Normal', 'Falla'])
    ax.set_xlabel('')

plt.suptitle('Distribución de sensores analógicos por clase', fontsize=14, fontweight='bold')
plt.tight_layout()

## 5. Correlación de features con la etiqueta de falla

In [ ]:
corr_failure = (df.drop(columns=['timestamp'])
                  .corr(numeric_only=True)['failure']
                  .drop('failure')
                  .sort_values(key=abs, ascending=False))

colors = ['tomato' if v > 0 else 'steelblue' for v in corr_failure]

plt.figure(figsize=(10, 6))
bars = plt.barh(corr_failure.index, corr_failure.values, color=colors, edgecolor='white')
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Correlación de Pearson con failure')
plt.title('Correlación de cada feature con la etiqueta de falla', fontweight='bold', fontsize=13)
plt.gca().invert_yaxis()
for bar, val in zip(bars, corr_failure.values):
    plt.text(val + (0.002 if val >= 0 else -0.002), bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=9)
plt.tight_layout()